<a href="https://colab.research.google.com/github/expaetra/CM3070_final_project/blob/master/02_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook we construct the training dataset for the discipline and field classification tasks.

The categories selected in the previous step are used to defien a taxonomy mapping table that will assign eah category to a CSO-aligned discipline and field label.

We then scrape papers from the arXiv API and collect a fixed number of abstracts per category to ensure class balance. The output of this stage is a clean, labeled CSV dataset ready for model training.

In [30]:
!git clone https://github.com/expaetra/CM3070_final_project.git
%cd CM3070_final_project

Cloning into 'CM3070_final_project'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 68 (delta 2), reused 64 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 5.28 MiB | 18.89 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/CM3070_final_project/CM3070_final_project/CM3070_final_project


In [31]:
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
import pandas as pd
import time
import requests

In [32]:
# load selected categories from repo

selected = pd.read_csv("backend/data/selected_categories.csv")
print(f"{len(selected)} categories loaded")
print(selected.to_string())

32 categories loaded
   category  total_papers
0     cs.LG        258190
1     cs.CV        185665
2     cs.AI        168181
3     cs.CL        104601
4     cs.IT         53521
5     cs.RO         50916
6     cs.CR         46546
7     cs.HC         29041
8     cs.DS         28222
9     cs.DC         27724
10    cs.CY         27354
11    cs.NI         26796
12    cs.SE         25302
13    cs.IR         24351
14    cs.SI         23069
15    cs.SD         20214
16    cs.LO         18901
17    cs.NE         17322
18    cs.DM         15372
19    cs.GT         14333
20    cs.CC         12592
21    cs.MA         11553
22    cs.DB         11369
23    cs.CE         10072
24    cs.PL          9537
25    cs.MM          9230
26    cs.CG          7907
27    cs.AR          7791
28    cs.ET          6962
29    cs.DL          5870
30    cs.FL          5866
31    cs.PF          5068


This section defines a taxonomy mapping table that assigns each arXiv category code a field and discipline label. It uses CSO as a reference to build a simplified two level hierarchy.

In [33]:
taxonomy = {
    'cs.LG': {'field': 'Machine Learning', 'discipline': 'Artificial Intelligence'},
    'cs.CV': {'field': 'Computer Vision', 'discipline': 'Computer Imaging and Vision'},
    'cs.AI': {'field': 'General Artificial Intelligence', 'discipline': 'Artificial Intelligence'},
    'cs.CL': {'field': 'Natural Language Processing', 'discipline': 'Artificial Intelligence'},
    'cs.IT': {'field': 'Information Theory', 'discipline': 'Communication'},
    'cs.RO': {'field': 'General Robotics', 'discipline': 'Robotics'},
    'cs.CR': {'field': 'Cryptography', 'discipline': 'Cybersecurity'},
    'cs.HC': {'field': 'General Human-Computer Interaction', 'discipline': 'Human-Computer Interaction'},
    'cs.DS': {'field': 'Data Structures and Algorithms', 'discipline': 'Theory of Computation'},
    'cs.DC': {'field': 'Distributed Computer Systems', 'discipline': 'Distributed Systems'},
    'cs.CY': {'field': 'Computers and Society', 'discipline': 'Social Computing'},
    'cs.NI': {'field': 'General Computer Networks', 'discipline': 'Computer Networks'},
    'cs.SE': {'field': 'General Software Engineering', 'discipline': 'Software Engineering'},
    'cs.IR': {'field': 'General Information Retrieval', 'discipline': 'Information Retrieval'},
    'cs.SI': {'field': 'Social Networks', 'discipline': 'World Wide Web'},
    'cs.SD': {'field': 'Sound and Signal Processing', 'discipline': 'Multimedia'},
    'cs.LO': {'field': 'Formal Logic', 'discipline': 'Artificial Intelligence'},
    'cs.NE': {'field': 'Neural Networks', 'discipline': 'Artificial Intelligence'},
    'cs.DM': {'field': 'Discrete Mathematics', 'discipline': 'Theory of Computation'},
    'cs.GT': {'field': 'Game Theory', 'discipline': 'Artificial Intelligence'},
    'cs.CC': {'field': 'Computational Complexity', 'discipline': 'Theory of Computation'},
    'cs.MA': {'field': 'Multiagent Systems', 'discipline': 'Artificial Intelligence'},
    'cs.DB': {'field': 'Database Systems', 'discipline': 'Computer Systems'},
    'cs.CE': {'field': 'Computational Engineering', 'discipline': 'Computational Science'},
    'cs.PL': {'field': 'Programming Languages', 'discipline': 'Computer Programming'},
    'cs.MM': {'field': 'General Multimedia', 'discipline': 'Multimedia'},
    'cs.CG': {'field': 'Computational Geometry', 'discipline': 'Computer Graphics'},
    'cs.AR': {'field': 'Hardware Architecture', 'discipline': 'Computer Hardware'},
    'cs.ET': {'field': 'Emerging Technologies', 'discipline': 'Emerging Technologies'},
    'cs.DL': {'field': 'Digital Libraries', 'discipline': 'World Wide Web'},
    'cs.FL': {'field': 'Formal Languages and Automata Theory', 'discipline': 'Theoretical Computer Science'},
    'cs.PF': {'field': 'Performance', 'discipline': 'Computer Systems'},
}

In [34]:
# inspect

selected['field'] = selected['category'].map({k: v['field'] for k, v in taxonomy.items()})
selected['discipline'] = selected['category'].map({k: v['discipline'] for k, v in taxonomy.items()})
print(selected.head())

  category  total_papers                            field  \
0    cs.LG        258190                 Machine Learning   
1    cs.CV        185665                  Computer Vision   
2    cs.AI        168181  General Artificial Intelligence   
3    cs.CL        104601      Natural Language Processing   
4    cs.IT         53521               Information Theory   

                    discipline  
0      Artificial Intelligence  
1  Computer Imaging and Vision  
2      Artificial Intelligence  
3      Artificial Intelligence  
4                Communication  


We query the arXiv API for each category and collect a fixed number of abstracts. Each record stores the abstract text alongside its category, field, and discipline labels. The output is a balanced, labeled dataset ready for model training.

In [35]:
# abstract collection

def scrape_arxiv(category, max_results=500, wait=3):
    abstracts=[]
    batch_size=100
    start=0

    while len(abstracts) <max_results:
        url = (
            f"https://export.arxiv.org/api/query?"
            f"search_query=cat:{category}&start={start}&max_results={batch_size}"
        )
        response = requests.get(url)
        root = ET.fromstring(response.content)
        entries =root.findall('{http://www.w3.org/2005/Atom}entry')
        if not entries:
            break

        # extract abstracts from the entries
        for entry in entries:
            abstract = entry.find('{http://www.w3.org/2005/Atom}summary')
            if abstract is not None:
                abstracts.append(abstract.text.strip())
            if len(abstracts) >= max_results:
                break

        # move to the next batch
        start+=batch_size
        time.sleep(wait)

    return abstracts

In [37]:
# loop through the categories to select abstracts

records = []

for _, row in selected.iterrows():
    category = row['category']
    field = row['field']
    discipline = row['discipline']
    print(f"Scraping {category}:")
    # get abstracts for category
    abstracts = scrape_arxiv(category, max_results=500)

    for abstract in abstracts:
        records.append({
            'category': category,
            'field': field,
            'discipline': discipline,
            'abstract':abstract
        })
    print(f"  {len(abstracts)} abstracts collected!")
print(f"\nTotal records: {len(records)}")

Scraping cs.LG:
  500 abstracts collected!
Scraping cs.CV:
  500 abstracts collected!
Scraping cs.AI:
  500 abstracts collected!
Scraping cs.CL:
  500 abstracts collected!
Scraping cs.IT:
  500 abstracts collected!
Scraping cs.RO:
  500 abstracts collected!
Scraping cs.CR:
  500 abstracts collected!
Scraping cs.HC:
  500 abstracts collected!
Scraping cs.DS:
  500 abstracts collected!
Scraping cs.DC:
  500 abstracts collected!
Scraping cs.CY:
  500 abstracts collected!
Scraping cs.NI:
  500 abstracts collected!
Scraping cs.SE:
  500 abstracts collected!
Scraping cs.IR:
  500 abstracts collected!
Scraping cs.SI:
  500 abstracts collected!
Scraping cs.SD:
  500 abstracts collected!
Scraping cs.LO:
  500 abstracts collected!
Scraping cs.NE:
  500 abstracts collected!
Scraping cs.DM:
  500 abstracts collected!
Scraping cs.GT:
  500 abstracts collected!
Scraping cs.CC:
  500 abstracts collected!
Scraping cs.MA:
  500 abstracts collected!
Scraping cs.DB:
  500 abstracts collected!
Scraping cs

In [38]:
# save

import os
os.makedirs("/content/drive/MyDrive/CM3070_final_project/backend/data", exist_ok=True)

# convert to DataFrame
df_abstracts = pd.DataFrame(records)

df_abstracts.to_csv("/content/drive/MyDrive/CM3070_final_project/backend/data/arxiv_abstracts.csv",index=False)

print(f"Saved {len(df_abstracts)} to backend/data/arxiv_abstracts.csv")

Saved 16000 to backend/data/arxiv_abstracts.csv
